# STEP 1 — Daily CAM7_L58_HR files with Tb, PRECT, X/Y and (y,x) dims

This notebook processes paired CAM HR `h5a` / `h5i` files and produces **one NetCDF file per day** with:

- `PRECT` converted from m/s to mm/hr
- `Tb` (flux equivalent brightness temperature) computed from `FLUT`
- 2‑D coordinates `X(y,x)` and `Y(y,x)`
- Horizontal dimensions renamed from `lat, lon` to `y, x`
- `time` as an **UNLIMITED** dimension

The focus is on being **robust and simple**, not fancy.


## 1. Imports and xarray options


In [1]:
%%time
import os
import glob
import gc
import warnings

import numpy as np
import xarray as xr

warnings.filterwarnings("ignore")
xr.set_options(keep_attrs=True)

print("Imports complete.")


Imports complete.
CPU times: user 254 ms, sys: 110 ms, total: 364 ms
Wall time: 623 ms


## 2. Locate input CAM HR h5a / h5i files

Adjust the glob patterns below if needed. They should point to the matching
`h5a` and `h5i` history files for your HR CAM run.


In [2]:
%%time
# TODO: adjust paths if needed
h5a = sorted(
    glob.glob(
        "/glade/campaign/cesm/community/amwg/cam_hr/"
        "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5a.*"
    )
)

h5i = sorted(
    glob.glob(
        "/glade/campaign/cesm/community/amwg/cam_hr/"
        "f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5i.*"
    )
)

print("Number of h5a files:", len(h5a))
print("Number of h5i files:", len(h5i))

if len(h5a) != len(h5i):
    raise ValueError("h5a and h5i file lists have different lengths!")


Number of h5a files: 512
Number of h5i files: 512
CPU times: user 6.57 ms, sys: 3.43 ms, total: 10 ms
Wall time: 11.9 ms


## 3. Define processing function

For each index `ifile` in `h5a`/`h5i`:

1. Open `h5a[ifile]`, keep `PRECT`, convert to mm/hr.
2. Open `h5i[ifile]`, compute `Tb` from `FLUT`.
3. Merge the two datasets.
4. Rename horizontal dimensions `lat -> y` and `lon -> x`.
5. Add 2‑D `X(y,x)` and `Y(y,x)` variables.
6. Split the 15‑day file into 15 daily files (24 time steps each).
7. Write each daily file as `CAM7_L58_HR_hourly.YYYY-MM-DD.nc` with `time` unlimited.


In [3]:
%%time
# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars = ["PRECT"]

# Output directory
outdir = "/glade/derecho/scratch/hannay/MOAPP/daily_files/"
os.makedirs(outdir, exist_ok=True)


def process_one_pair(ifile: int):
    """Process one (h5a, h5i) pair into daily CAM7_L58_HR_hourly.YYYY-MM-DD.nc files.

    Steps:
    - Convert PRECT to mm/hr
    - Compute Tb from FLUT
    - Merge
    - Rename dims lat->y, lon->x
    - Add X(y,x) and Y(y,x)
    - Split into 24-step daily files with unlimited time
    """

    print("====")
    print(f"Processing index {ifile}")
    print("  h5a:", h5a[ifile])
    print("  h5i:", h5i[ifile])

    # --- Open h5a and convert PRECT ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time": 24}, engine="netcdf4")
    h5adat = h5adat[h5avars]

    prect_attrs = h5adat["PRECT"].attrs
    h5adat["PRECT"] = h5adat["PRECT"] * 1000.0 * 3600.0  # m/s -> mm/hr
    h5adat["PRECT"].attrs = prect_attrs
    h5adat["PRECT"].attrs["units"] = "mm hr-1"

    # --- Open h5i and compute Tb ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time": 24}, engine="netcdf4")
    FLUT = h5idat["FLUT"]

    print("  FLUT min/max:", float(FLUT.min()), float(FLUT.max()))

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)

    Tb = Tb.rename("Tb")
    Tb.attrs["long_name"] = "flux equivalent brightness temperature"
    Tb.attrs["units"] = "K"

    # Simple, explicit encoding for Tb
    Tb = Tb.astype("float32")
    Tb.encoding = {}
    Tb.encoding["_FillValue"] = np.float32(np.nan)

    h5idat["Tb"] = Tb

    print("  Tb TEST min/max:", float(Tb.min()), float(Tb.max()))

    # --- Merge using the h5i time axis ---
    h5adat["time"] = h5idat.time.values
    newdat = xr.merge([h5adat, h5idat])

    print("  Tb after merge min/max:",
          float(newdat["Tb"].min()), float(newdat["Tb"].max()))

    # --- Rename spatial dimensions lat->y, lon->x ---
    if "lat" not in newdat.dims or "lon" not in newdat.dims:
        raise KeyError("Expected dims 'lat' and 'lon' in merged dataset.")

    newdat = newdat.rename({"lat": "y", "lon": "x"})

    # After rename, coords 'lat' and 'lon' now have dims ('y',) and ('x',)
    if "lat" not in newdat.coords or "lon" not in newdat.coords:
        raise KeyError("Expected coord variables 'lat' and 'lon' after renaming dims.")

    lat_1d = newdat["lat"]  # dims: (y,)
    lon_1d = newdat["lon"]  # dims: (x,)

    # --- Build 2D X(y,x) and Y(y,x) ---
    if "Y" not in newdat.variables or "X" not in newdat.variables:
        Y_2d, X_2d = xr.broadcast(lat_1d, lon_1d)
        Y_2d = Y_2d.rename("Y")
        X_2d = X_2d.rename("X")

        Y_2d.attrs["long_name"] = "latitude on (y,x) grid"
        Y_2d.attrs["units"] = lat_1d.attrs.get("units", "degrees_north")

        X_2d.attrs["long_name"] = "longitude on (y,x) grid"
        X_2d.attrs["units"] = lon_1d.attrs.get("units", "degrees_east")

        newdat["Y"] = Y_2d
        newdat["X"] = X_2d

    # Optional: set fill values for 1D lat/lon coords
    newdat["lat"].encoding["_FillValue"] = np.nan
    newdat["lon"].encoding["_FillValue"] = np.nan

    # --- Split into daily chunks by index (24 steps per day) ---
    written_files = []

    nt = newdat.dims["time"]
    if nt % 24 != 0:
        raise RuntimeError(f"Time length {nt} is not a multiple of 24.")

    ndays = nt // 24
    print(f"  Found {nt} timesteps -> {ndays} days (24 hours each)")

    for day_idx in range(ndays):
        t0 = day_idx * 24
        t1 = t0 + 24

        day_ds = newdat.isel(time=slice(t0, t1))

        # Use first time in the day for filename
        t0_val = day_ds["time"].values[0]
        date_str = f"{t0_val.year:04d}-{t0_val.month:02d}-{t0_val.day:02d}"

        outpath_day = os.path.join(outdir, f"CAM7_L58_HR_hourly.{date_str}.nc")

        if os.path.exists(outpath_day):
            print(f"  [SKIP] File already exists: {outpath_day}")
            continue

        tb_min = float(day_ds["Tb"].min())
        tb_max = float(day_ds["Tb"].max())
        print(f"    Day {date_str}: Tb min/max = {tb_min:.3f}, {tb_max:.3f}")

        if tb_min == 0.0 and tb_max == 0.0:
            raise RuntimeError(
                f"Tb is all zero in daily slice {date_str} BEFORE writing."
            )

        # Write with time as unlimited dimension
        day_ds.to_netcdf(outpath_day, format="NETCDF4", unlimited_dims="time")

        written_files.append(outpath_day)

    # --- Cleanup ---
    newdat.close()
    h5adat.close()
    h5idat.close()

    del newdat, h5adat, h5idat, FLUT, Tf, Tb, day_ds
    gc.collect()

    return written_files


CPU times: user 687 μs, sys: 261 μs, total: 948 μs
Wall time: 5.44 ms


## 4. Test on a single file

Run this once to make sure everything works and inspect the output.


In [ ]:
%%time
if len(h5a) > 0:
    test_written = process_one_pair(0)
    print("Test written files:", test_written)
else:
    print("No h5a files found; adjust the glob pattern above.")


====
Processing index 0
  h5a: /glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.cam.h5a.1990-01-01-03600.nc
  h5i: /glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001.cam.h5i.1990-01-01-03600.nc
  FLUT min/max: 0.0 419.1314392089844
  Tb TEST min/max: -0.0 347.5984191894531


## 5. Process all files (serial loop)

This loop runs `process_one_pair` over all indices. It will skip daily files
that already exist.


In [ ]:
%%time
all_written = []
nfiles = len(h5a)
print("Total input pairs:", nfiles)

for i in range(nfiles):
    try:
        files_i = process_one_pair(i)
        all_written.extend(files_i)
    except Exception as e:
        print(f"[ERROR] Failed on index {i}: {e}")

print("Done. Number of daily files written in this run:", len(all_written))
